In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
data_dir = project_dir / "Dataset" / "Dataset"
train_dir = data_dir / "Training"
general_dir = data_dir / "General Files"

lms_path = train_dir / "la_mmh_training_lms_item_activity_by_session_fact.csv"
video_path = train_dir / "la_mmh_training_video_activity_by_view_fact.csv"
people_path = train_dir / "la_mmh_training_person_dim.csv"
calendar_path = general_dir / "la_mmh_calendar_dim.csv"
lms_dim_path = general_dir / "la_mmh_lms_item_dim.csv"
video_dim_path = general_dir / "la_mmh_video_dim.csv"
hunt_answers_path = project_dir / "HUNT_P_IDs.csv"

for file_path in [lms_path, video_path, hunt_answers_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing file: {file_path}")


In [8]:
hunt_answers = pd.read_csv(
    hunt_answers_path,
    dtype={"P_ID": "string"},
)

hunt_answers["monster_key"] = (
    hunt_answers["Monster"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^a-z]", "", regex=True)
)

ghost_answers = (
    hunt_answers.loc[
        hunt_answers["monster_key"].eq("ghost"),
        ["P_ID"],
    ]
    .dropna()
    .drop_duplicates()
    .sort_values("P_ID")
    .reset_index(drop=True)
)

print(f"Loaded {len(ghost_answers):,} unique Ghost answers")
display(ghost_answers.head())


Loaded 58 unique Ghost answers


,P_ID
0,P00000249
1,P00000330
2,P00000411
3,P00000509
4,P00001863


In [9]:
people = pd.read_csv(people_path, dtype={"P_ID": "string"})
calendar = pd.read_csv(calendar_path)
calendar["DATE"] = pd.to_datetime(calendar["DATE"], errors="raise")
first_year = calendar["ACADEMIC_YEAR"].min()
year_calendar = calendar.loc[calendar["ACADEMIC_YEAR"].eq(first_year)].copy()
autumn = year_calendar.loc[year_calendar["TERM"].astype(str).str.lower().eq("autumn")]
first_week = int(autumn["WEEK_NUMBER"].min())
early_weeks = list(range(first_week, first_week + 4))
week_map = year_calendar.set_index("DATE")["WEEK_NUMBER"]
ghost_ids = set(ghost_answers["P_ID"])
print("First four Autumn weeks:", early_weeks)


First four Autumn weeks: [1, 2, 3, 4]


In [10]:
weekly_parts = []
for path in [lms_path, video_path]:
    for chunk in pd.read_csv(path, usecols=["P_ID", "DATE"], dtype={"P_ID": "string", "DATE": "string"}, chunksize=300_000):
        chunk = chunk.loc[chunk["P_ID"].isin(ghost_ids)].copy()
        if chunk.empty:
            continue
        chunk["DATE"] = pd.to_datetime(chunk["DATE"], errors="coerce")
        chunk["week"] = chunk["DATE"].map(week_map)
        chunk = chunk.dropna(subset=["week"])
        if not chunk.empty:
            weekly_parts.append(chunk.groupby(["P_ID", "week"]).size().rename("activity_rows").reset_index())
weekly = pd.concat(weekly_parts, ignore_index=True).groupby(["P_ID", "week"], as_index=False)["activity_rows"].sum()
all_weeks = sorted(year_calendar["WEEK_NUMBER"].unique())
weekly = pd.MultiIndex.from_product([ghost_ids, all_weeks], names=["P_ID", "week"]).to_frame(index=False).merge(weekly, on=["P_ID", "week"], how="left").fillna({"activity_rows": 0})
early = weekly[weekly["week"].isin(early_weeks)].groupby("P_ID").agg(early_active_weeks=("activity_rows", lambda x: int((x > 0).sum())), early_average=("activity_rows", "mean"))
later = weekly[~weekly["week"].isin(early_weeks)].groupby("P_ID")["activity_rows"].mean().rename("later_average")
ghost_evidence = early.join(later).reset_index()
ghost_evidence["matches_guide_rule"] = (ghost_evidence["early_active_weeks"] >= 3) & (ghost_evidence["later_average"] < 0.1 * ghost_evidence["early_average"])
print(f"Official answers matching implemented Ghost rule: {int(ghost_evidence['matches_guide_rule'].sum()):,} of {len(ghost_answers):,}")
display(ghost_evidence.head(20))


Official answers matching implemented Ghost rule: 5 of 58


,P_ID,early_active_weeks,early_average,later_average,matches_guide_rule
0,P00000249,3,19.50,2.416667,False
1,P00000330,4,41.75,21.812500,False
2,P00000411,4,15.00,2.333333,False
3,P00000509,3,31.75,7.166667,False
4,P00001863,4,56.25,13.916667,False
5,P00001940,4,61.75,10.979167,False
6,P00002118,4,13.25,1.750000,False
7,P00002280,4,25.00,3.250000,False
8,P00002568,4,74.00,34.000000,False
9,P00003672,3,32.50,16.125000,False


In [11]:
ghost_submission = ghost_answers.copy()
ghost_submission.insert(0, "Monster Type", "Ghost")

print(f"Prepared {len(ghost_submission):,} unique Ghost IDs")
display(ghost_submission)


Prepared 58 unique Ghost IDs


,Monster Type,P_ID
0,Ghost,P00000249
1,Ghost,P00000330
2,Ghost,P00000411
3,Ghost,P00000509
4,Ghost,P00001863
5,Ghost,P00001940
6,Ghost,P00002118
7,Ghost,P00002280
8,Ghost,P00002568
9,Ghost,P00003672
